In [ ]:
# ============================================================
# AI Chatbot - Google Colab
# Built with Groq API (Free) + Gradio UI
# ============================================================

# ---------- STEP 1: Install Dependencies ----------
!pip install groq gradio -q

# ---------- STEP 2: Import Libraries ----------
import gradio as gr
from groq import Groq
import os

# ---------- STEP 3: Setup API Key ----------
# Option A: Enter manually (simple)
GROQ_API_KEY = "your_groq_api_key_here"
# SAFE METHOD — key stays private in Colab Secrets
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')


# ---------- STEP 4: Initialize Groq Client ----------
client = Groq(api_key=GROQ_API_KEY)

# ---------- STEP 5: Chatbot Logic ----------
def chat(user_message, history, system_prompt, model_choice, temperature):
    """
    Send message to AI and return response.
    history: list of [user_msg, bot_msg] pairs (Gradio format)
    """
    # Build message history for API
    messages = []

    # Add system prompt
    if system_prompt.strip():
        messages.append({"role": "system", "content": system_prompt})

    # Add conversation history
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        if bot_msg:
            messages.append({"role": "assistant", "content": bot_msg})

    # Add current user message
    messages.append({"role": "user", "content": user_message})

    try:
        # Call Groq API
        response = client.chat.completions.create(
            model=model_choice,
            messages=messages,
            temperature=temperature,
            max_tokens=1024,
        )
        return response.choices[0].message.content

    except Exception as e:
        return f"Error: {str(e)}"


def respond(user_message, history, system_prompt, model_choice, temperature):
    """Gradio chatbot handler."""
    if not user_message.strip():
        return history, ""

    bot_response = chat(user_message, history, system_prompt, model_choice, temperature)
    history.append([user_message, bot_response])
    return history, ""


def clear_chat():
    """Clear conversation history."""
    return [], ""


# ---------- STEP 6: Gradio UI ----------
MODELS = [
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
    "mixtral-8x7b-32768",
    "gemma2-9b-it",
]

DEFAULT_SYSTEM = "You are a helpful, friendly, and knowledgeable AI assistant. Answer clearly and concisely."

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="AI Chatbot",
    css="""
    .chatbot { height: 480px; }
    #title { text-align: center; margin-bottom: 10px; }
    """
) as demo:

    gr.Markdown("# AI Chatbot\nPowered by **Groq** · Fast & Free", elem_id="title")

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Conversation",
                elem_classes=["chatbot"],
                bubble_full_width=False,
                show_label=False,
            )
            with gr.Row():
                msg_input = gr.Textbox(
                    placeholder="Type your message and press Enter...",
                    show_label=False,
                    scale=5,
                    container=False,
                )
                send_btn = gr.Button("Send ", variant="primary", scale=1)

            clear_btn = gr.Button("Clear Chat", variant="secondary")

        with gr.Column(scale=1):
            gr.Markdown("### Settings")
            model_dropdown = gr.Dropdown(
                choices=MODELS,
                value=MODELS[0],
                label="Model",
            )
            temperature_slider = gr.Slider(
                minimum=0.0,
                maximum=1.5,
                value=0.7,
                step=0.1,
                label="Temperature",
                info="Higher = more creative",
            )
            system_prompt_box = gr.Textbox(
                value=DEFAULT_SYSTEM,
                label="System Prompt",
                lines=5,
                info="Define the AI's personality/role",
            )

            gr.Markdown("""
            ### Tips
            - **Llama 3.3 70B** → Best quality
            - **Llama 3.1 8B** → Fastest
            - **Mixtral** → Great reasoning
            - Adjust **temperature** to control creativity
            """)

    # ---------- Event Handlers ----------
    state = gr.State([])

    msg_input.submit(
        respond,
        inputs=[msg_input, state, system_prompt_box, model_dropdown, temperature_slider],
        outputs=[state, msg_input],
    ).then(lambda h: h, state, chatbot)

    send_btn.click(
        respond,
        inputs=[msg_input, state, system_prompt_box, model_dropdown, temperature_slider],
        outputs=[state, msg_input],
    ).then(lambda h: h, state, chatbot)

    clear_btn.click(clear_chat, outputs=[state, msg_input]).then(
        lambda: [], None, chatbot
    )

# ---------- STEP 7: Launch ----------
demo.launch(
    share=True,         # Creates public URL (works in Colab)
    debug=False,
    show_error=True,
)